# Demand Modeling and Net Flow Derivation — Gold V2 (XGBoost)

**Overview**
This notebook implements a modeling workflow using Gold V2 data to estimate station-level demand signals.

The process includes training XGBoost models for departures and arrivals, followed by the derivation of net flow as an indicator of station imbalance.

**Objective**
The goal is to model bike demand patterns and derive imbalance metrics that reflect the difference between incoming and outgoing trips at each station.

**Data Source**
- Dataset: Gold V2 integrated dataset
- Granularity: station-hour level
- Includes temporal, behavioral, weather, and event features

In [0]:
# %pip install xgboost==2.0.3       
# %restart_python

## Train DEP + ARR from GOLD_V2 (integrated dataset) - FINAL

In [0]:
# ============================================================
# Train DEP + ARR from GOLD_V2 (integrated dataset)
# Step A: Build temporal features (lags + rolling) and store once
# Step B: Progressive monthly train/eval using Hash Compact encoding
# Serverless-safe: no persist/cache, 1 toPandas per month (cache)
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

import pandas as pd
import numpy as np
import zlib

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------------------------------------------
# 0) PATHS
# ------------------------------------------------------------
GOLD_V2_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/gold/gold_v2_spatiotemporal_events"

# Feature store derived from GOLD_V2 (version it!)
FEATURES_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/features/goldv2_features_v1"

# Eval outputs (version them!)
EVAL_DIR_DEP = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/goldv2_dep_hashcompact_v1"
EVAL_DIR_ARR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/goldv2_arr_hashcompact_v1"

REBUILD_FEATURES = False   # <-- set True once to build FEATURES_DIR
CACHE_FEATURES   = False   # serverless safe (not used)

# ------------------------------------------------------------
# 1) CONFIG
# ------------------------------------------------------------
# Downtown bounding box
DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX = 43.63, 43.67
DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX = -79.41, -79.37

TRAIN_LOOKBACK_DAYS = 90

HASH_BUCKETS = 512
BUCKET_COL = "station_bucket"

N_ESTIMATORS_GRID = [300, 500, 700, 900, 1100]

base_xgb_params = dict(
    n_estimators=600,       # overwritten by grid
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:squarederror",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

# Evaluate full period Oct-2022 to Sep-2024 (your dataset range)
START_Y, START_M = 2022, 10
END_Y, END_M     = 2024,  9

# ------------------------------------------------------------
# 2) HELPERS
# ------------------------------------------------------------
def path_exists(path: str) -> bool:
    try:
        _ = dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def station_to_bucket(station_id: str, n_buckets: int) -> int:
    if station_id is None:
        return 0
    b = str(station_id).encode("utf-8")
    return zlib.crc32(b) % n_buckets

def prev_months_in_lookback(test_start: pd.Timestamp, lookback_days: int):
    start = (test_start - pd.Timedelta(days=lookback_days)).to_period("M")
    end = (test_start - pd.Timedelta(days=1)).to_period("M")
    periods = pd.period_range(start, end, freq="M")
    return [(int(p.year), int(p.month)) for p in periods]

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def in_range(y, m, sy, sm, ey, em) -> bool:
    return (y > sy or (y == sy and m >= sm)) and (y < ey or (y == ey and m <= em))

def train_best_model(train_X, train_y, test_X, test_y, n_grid):
    best_rmse_val = np.inf
    best_mae_val = None
    best_pred = None
    best_n = None

    for n_est in n_grid:
        params = dict(base_xgb_params)
        params["n_estimators"] = int(n_est)

        model = XGBRegressor(**params)
        model.fit(train_X, train_y)

        pred = model.predict(test_X).astype(np.float32)
        cur_rmse = rmse(test_y, pred)

        if cur_rmse < best_rmse_val:
            best_rmse_val = cur_rmse
            best_mae_val = float(mean_absolute_error(test_y, pred))
            best_pred = pred
            best_n = int(n_est)

    return best_pred, best_mae_val, best_rmse_val, best_n

# ------------------------------------------------------------
# 3) BUILD FEATURES FROM GOLD_V2 (lags + rolling) -> FEATURES_DIR
# ------------------------------------------------------------
if REBUILD_FEATURES:
    if not path_exists(GOLD_V2_DIR):
        raise Exception(f"GOLD_V2_DIR not found: {GOLD_V2_DIR}")

    print("REBUILD_FEATURES=True -> building features from:", GOLD_V2_DIR)
    df = spark.read.parquet(GOLD_V2_DIR)

    # Downtown filter (lat/lon)
    df = df.filter(
        (F.col("lat").between(DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX)) &
        (F.col("lon").between(DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX))
    )

    print("Stations (downtown):", df.select("station_id").distinct().count())
    print("Rows (downtown):", df.count())

    # Create date and dow_num (Spark dayofweek: Sun=1..Sat=7)
    df = (
        df
        .withColumn("date", F.make_date("year", "month", "day"))
        .withColumn("dow_num", F.dayofweek("date"))
    )

    # Window by station ordered by time
    w = Window.partitionBy("station_id").orderBy(F.col("date"), F.col("hour"))

    # Lags for departures + arrivals
    df = (
        df
        .withColumn("lag1_dep",   F.lag("departures", 1).over(w))
        .withColumn("lag2_dep",   F.lag("departures", 2).over(w))
        .withColumn("lag24_dep",  F.lag("departures", 24).over(w))
        .withColumn("lag168_dep", F.lag("departures", 168).over(w))

        .withColumn("lag1_arr",   F.lag("arrivals", 1).over(w))
        .withColumn("lag2_arr",   F.lag("arrivals", 2).over(w))
        .withColumn("lag24_arr",  F.lag("arrivals", 24).over(w))
        .withColumn("lag168_arr", F.lag("arrivals", 168).over(w))
    )

    # Rolling windows (past only)
    roll_w_3h  = w.rowsBetween(-3,  -1)
    roll_w_24h = w.rowsBetween(-24, -1)

    df = (
        df
        .withColumn("roll_mean_3h_dep",  F.avg("departures").over(roll_w_3h))
        .withColumn("roll_std_24h_dep",  F.stddev("departures").over(roll_w_24h))

        .withColumn("roll_mean_3h_arr",  F.avg("arrivals").over(roll_w_3h))
        .withColumn("roll_std_24h_arr",  F.stddev("arrivals").over(roll_w_24h))
    )

    # Drop rows without sufficient history
    df_feat = df.dropna()

    (df_feat.write
        .mode("overwrite")
        .partitionBy("year", "month")
        .parquet(FEATURES_DIR))

    print("Features written to:", FEATURES_DIR)

# ------------------------------------------------------------
# 4) LOAD FEATURES
# ------------------------------------------------------------
if not path_exists(FEATURES_DIR):
    raise Exception(
        f"FEATURES_DIR not found: {FEATURES_DIR}\n"
        "Set REBUILD_FEATURES=True once to build features, then set it back to False."
    )

df_feat = spark.read.parquet(FEATURES_DIR)
loaded_rows = df_feat.count()
print("Loaded feature rows:", f"{loaded_rows:,}")

# ------------------------------------------------------------
# 5) Define columns for training (avoid arrays!)
# ------------------------------------------------------------
# Weather
weather_cols = ["temperature_2m_celsius", "apparent_temperature_celsius"]

# Event numeric columns (from your GOLD_V2 build)
event_numeric_cols = [
    "event_day_flag",
    "event_day_attendance_sum",
    "events_day_count",
    "event_active_nearby_flag",
    "events_nearby_count",
    "nearest_event_km",
    "event_weighted_intensity",
    "event_attendance_est_sum_nearby",
    "event_impact_score",
]

# Calendar/base
base_cols = ["station_id", "year", "month", "day", "hour", "date", "dow_num"]

# Targets
DEP_TGT = "departures"
ARR_TGT = "arrivals"

# Dep temporal features (from REBUILD step)
dep_hist = ["lag1_dep","lag2_dep","lag24_dep","lag168_dep","roll_mean_3h_dep","roll_std_24h_dep"]
arr_hist = ["lag1_arr","lag2_arr","lag24_arr","lag168_arr","roll_mean_3h_arr","roll_std_24h_arr"]

# Fail-fast: ensure needed columns exist in Spark
need_cols = set(base_cols + weather_cols + event_numeric_cols + dep_hist + arr_hist + [DEP_TGT, ARR_TGT])
missing = [c for c in need_cols if c not in df_feat.columns]
if missing:
    raise Exception(f"Feature dataset missing columns: {missing}")
print("Column check passed")

# Model features (hash compact + calendar + weather + event + temporal)
common_feats = [
    BUCKET_COL, "month", "hour", "dow_num", "is_weekend",
    "temperature_2m_celsius", "apparent_temperature_celsius",
    "event_day_flag", "event_day_attendance_sum", "events_day_count",
    "event_active_nearby_flag", "events_nearby_count", "nearest_event_km",
    "event_weighted_intensity", "event_attendance_est_sum_nearby",
    "event_impact_score",
]

dep_model_features = common_feats + dep_hist
arr_model_features = common_feats + arr_hist

# ------------------------------------------------------------
# 6) Month list (full period)
# ------------------------------------------------------------
months_rows = df_feat.select("year", "month").distinct().collect()
months_list = sorted([(int(r["year"]), int(r["month"])) for r in months_rows])
months_list = [(y,m) for (y,m) in months_list if in_range(y,m, START_Y, START_M, END_Y, END_M)]
print("Months to evaluate:", len(months_list), "|", months_list[0], "->", months_list[-1])

# ------------------------------------------------------------
# 7) Pandas month cache (1 toPandas per month)
# ------------------------------------------------------------
month_cache = {}

select_cols_for_pandas = (
    base_cols
    + weather_cols
    + event_numeric_cols
    + dep_hist
    + arr_hist
    + [DEP_TGT, ARR_TGT]
)

def load_month_pd(y:int, m:int) -> pd.DataFrame:
    key = (y,m)
    if key in month_cache:
        return month_cache[key]

    sdf = (df_feat
        .filter((F.col("year")==y) & (F.col("month")==m))
        .select(select_cols_for_pandas)
    )

    pdf = sdf.toPandas()
    if len(pdf) > 0:
        pdf["date"] = pd.to_datetime(pdf["date"])

        # Hash bucket
        pdf[BUCKET_COL] = pdf["station_id"].map(lambda s: station_to_bucket(s, HASH_BUCKETS)).astype(np.int16)

        # Weekend flag (Spark convention: Sun=1, Sat=7)
        pdf["is_weekend"] = pdf["dow_num"].isin([1,7]).astype(np.int8)

        # Fill numeric nulls defensively
        for c in weather_cols + event_numeric_cols + dep_hist + arr_hist:
            if c in pdf.columns:
                pdf[c] = pdf[c].fillna(0)

        # nearest_event_km: sentinel 999.0 already exists for no event,
        # keep as is (it's informative)
    month_cache[key] = pdf
    return pdf

def build_xy(pdf: pd.DataFrame, features: list, target: str):
    missing = [c for c in features + [target] if c not in pdf.columns]
    if missing:
        raise KeyError(f"Missing cols in pandas for target='{target}': {missing}")
    X = pdf[features].astype(np.float32).values
    y = pdf[target].astype(np.float32).values
    return X, y

# ------------------------------------------------------------
# 8) Progressive monthly training/eval (DEP + ARR)
# ------------------------------------------------------------
results_dep = []
results_arr = []

for (y, m) in months_list:
    test_start = pd.Timestamp(year=y, month=m, day=1)
    test_end   = test_start + pd.offsets.MonthEnd(0)

    train_end   = test_start - pd.Timedelta(days=1)
    train_start = train_end - pd.Timedelta(days=TRAIN_LOOKBACK_DAYS)

    test_pd = load_month_pd(y, m)
    if test_pd.empty:
        continue

    # Build train from lookback months
    train_parts = []
    for (yy, mm) in prev_months_in_lookback(test_start, TRAIN_LOOKBACK_DAYS):
        if not in_range(yy, mm, START_Y, START_M, END_Y, END_M):
            continue
        part = load_month_pd(yy, mm)
        if not part.empty:
            train_parts.append(part)

    if not train_parts:
        continue

    train_all = pd.concat(train_parts, ignore_index=True)

    train_pd = train_all[(train_all["date"] >= train_start) & (train_all["date"] <= train_end)]
    test_pd_f = test_pd[(test_pd["date"] >= test_start) & (test_pd["date"] <= test_end)]

    if train_pd.empty or test_pd_f.empty:
        continue

    # Align ordering for hygiene
    join_keys = ["station_id","date","hour"]
    train_pd = train_pd.sort_values(join_keys).reset_index(drop=True)
    test_pd_f = test_pd_f.sort_values(join_keys).reset_index(drop=True)

    # ---------------- DEP ----------------
    dep_train_X, dep_train_y = build_xy(train_pd, dep_model_features, DEP_TGT)
    dep_test_X, dep_test_y   = build_xy(test_pd_f, dep_model_features, DEP_TGT)

    dep_baseline = test_pd_f["lag1_dep"].astype(np.float32).values
    dep_baseline_mae = float(mean_absolute_error(dep_test_y, dep_baseline))
    dep_baseline_rmse = rmse(dep_test_y, dep_baseline)

    dep_pred, dep_mae, dep_rmse, dep_best_n = train_best_model(
        dep_train_X, dep_train_y, dep_test_X, dep_test_y, N_ESTIMATORS_GRID
    )
    dep_impr = (dep_baseline_mae - dep_mae) / dep_baseline_mae * 100 if dep_baseline_mae else np.nan

    results_dep.append({
        "year": y, "month": m,
        "rows_test": int(len(dep_test_y)),
        "baseline_mae": dep_baseline_mae,
        "model_mae": float(dep_mae),
        "baseline_rmse": dep_baseline_rmse,
        "model_rmse": float(dep_rmse),
        "improvement_pct": float(dep_impr),
        "best_n_estimators": int(dep_best_n),
        "hash_buckets": int(HASH_BUCKETS),
        "train_lookback_days": int(TRAIN_LOOKBACK_DAYS)
    })

    # ---------------- ARR ----------------
    arr_train_X, arr_train_y = build_xy(train_pd, arr_model_features, ARR_TGT)
    arr_test_X, arr_test_y   = build_xy(test_pd_f, arr_model_features, ARR_TGT)

    arr_baseline = test_pd_f["lag1_arr"].astype(np.float32).values
    arr_baseline_mae = float(mean_absolute_error(arr_test_y, arr_baseline))
    arr_baseline_rmse = rmse(arr_test_y, arr_baseline)

    arr_pred, arr_mae, arr_rmse, arr_best_n = train_best_model(
        arr_train_X, arr_train_y, arr_test_X, arr_test_y, N_ESTIMATORS_GRID
    )
    arr_impr = (arr_baseline_mae - arr_mae) / arr_baseline_mae * 100 if arr_baseline_mae else np.nan

    results_arr.append({
        "year": y, "month": m,
        "rows_test": int(len(arr_test_y)),
        "baseline_mae": arr_baseline_mae,
        "model_mae": float(arr_mae),
        "baseline_rmse": arr_baseline_rmse,
        "model_rmse": float(arr_rmse),
        "improvement_pct": float(arr_impr),
        "best_n_estimators": int(arr_best_n),
        "hash_buckets": int(HASH_BUCKETS),
        "train_lookback_days": int(TRAIN_LOOKBACK_DAYS)
    })

# ------------------------------------------------------------
# 9) Show + Save
# ------------------------------------------------------------
dep_pd = pd.DataFrame(results_dep).sort_values(["year","month"])
arr_pd = pd.DataFrame(results_arr).sort_values(["year","month"])

print("DEP months evaluated:", len(dep_pd))
print("ARR months evaluated:", len(arr_pd))

display(dep_pd)
display(arr_pd)

spark.createDataFrame(dep_pd).write.mode("overwrite").parquet(EVAL_DIR_DEP)
spark.createDataFrame(arr_pd).write.mode("overwrite").parquet(EVAL_DIR_ARR)

print("Saved DEP eval to:", EVAL_DIR_DEP)
print("Saved ARR eval to:", EVAL_DIR_ARR)

**Net Flow Derivation and Artifact Preparation**

Predicted departures and arrivals are combined to derive net flow:

net_flow = arrivals - departures

This metric represents station-level imbalance.

Additionally, model artifacts and feature metadata are prepared for downstream use in prediction workflows.

## WEIGHT = 10 con JSON - FINAL

In [0]:
from pyspark.sql import functions as F

import pandas as pd
import numpy as np
import zlib
import json
from datetime import datetime, timezone

from xgboost import XGBRegressor
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ============================================================
# Net Flow (Derived) - GOLD_V2 Features (Weighted Training)
#
# Serverless-safe:
# - 1 toPandas por mes (cache en dict)
# - Grid controlado para n_estimators
# - Baseline net = lag1_arr - lag1_dep
# - Weighted training for Event hours via sample_weight
#
# Serving (Serverless-safe):
# - Train FINAL dep/arr models on ALL available data
# - Save models to JSON in DBFS Volumes using atomic writes
# - Save feature list metadata (JSON) for safe scoring
# - Validate artifacts (contract + boosters) after write
# ============================================================

# ------------------------------------------------------------
# 0) Paths
# ------------------------------------------------------------
FEATURES_GOLDV2_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/features/goldv2_features_v1"

USE_BEST_N_FROM_PREV_RUNS = False
EVAL_DEP_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/goldv2_dep_hashcompact_v1"
EVAL_ARR_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/goldv2_arr_hashcompact_v1"

WEIGHT_EVENT = 10
EVENT_FLAG_COL = "event_active_nearby_flag"

EVAL_DIR_NETFLOW = (
    f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/"
    f"goldv2_netflow_derived_hashcompact_v_final_weight{WEIGHT_EVENT}"
)

# Serving model outputs (DBFS Volumes)
SERVING_MODEL_DIR_DBFS = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/models"
DEP_MODEL_JSON_DBFS = f"{SERVING_MODEL_DIR_DBFS}/dep_weight{WEIGHT_EVENT}.json"
ARR_MODEL_JSON_DBFS = f"{SERVING_MODEL_DIR_DBFS}/arr_weight{WEIGHT_EVENT}.json"

SERVING_META_DIR_DBFS = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/metadata"
FEATURE_META_DIR_DBFS = f"{SERVING_META_DIR_DBFS}/features_weight{WEIGHT_EVENT}"
FEATURE_META_JSON_DBFS = f"{FEATURE_META_DIR_DBFS}/features.json"

# Safety caps for reading DBFS heads
MAX_MODEL_BYTES = 30_000_000   # 30MB
MAX_META_BYTES  = 500_000      # 0.5MB

# ------------------------------------------------------------
# 1) Config
# ------------------------------------------------------------
TRAIN_LOOKBACK_DAYS = 90

HASH_BUCKETS = 512
BUCKET_COL = "station_bucket"

N_ESTIMATORS_GRID = [300, 500, 700, 900, 1100]

base_xgb_params = dict(
    n_estimators=600,       # overwritten by grid / final
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:squarederror",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

# ------------------------------------------------------------
# 2) Helpers
# ------------------------------------------------------------
def path_exists(path: str) -> bool:
    try:
        _ = dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def ensure_dir_dbfs(dir_path: str):
    dbutils.fs.mkdirs(dir_path)

def atomic_put_text(dbfs_path: str, text: str):
    """
    Atomic-ish write pattern for DBFS:
    1) write to .tmp
    2) remove final if exists
    3) move tmp -> final
    """
    dst_dir = dbfs_path.rsplit("/", 1)[0]
    ensure_dir_dbfs(dst_dir)

    tmp = dbfs_path + ".tmp"
    dbutils.fs.put(tmp, text, True)

    try:
        dbutils.fs.rm(dbfs_path, True)
    except Exception:
        pass

    dbutils.fs.mv(tmp, dbfs_path, True)

def read_dbfs_text(dbfs_path: str, max_bytes: int) -> str:
    return dbutils.fs.head(dbfs_path, max_bytes)

def station_to_bucket(station_id: str, n_buckets: int) -> int:
    if station_id is None:
        return 0
    b = str(station_id).encode("utf-8")
    return zlib.crc32(b) % n_buckets

def prev_months_in_lookback(test_start: pd.Timestamp, lookback_days: int):
    start = (test_start - pd.Timedelta(days=lookback_days)).to_period("M")
    end = (test_start - pd.Timedelta(days=1)).to_period("M")
    periods = pd.period_range(start, end, freq="M")
    return [(int(p.year), int(p.month)) for p in periods]

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def build_xy(pdf: pd.DataFrame, features: list, target: str):
    missing = [c for c in features + [target] if c not in pdf.columns]
    if missing:
        raise KeyError(f"Missing columns for target='{target}': {missing}")
    X = pdf[features].astype(np.float32).values
    y = pdf[target].astype(np.float32).values
    return X, y

def build_sample_weights(pdf: pd.DataFrame, weight_event: float, event_col: str):
    w = np.ones(len(pdf), dtype=np.float32)
    if event_col in pdf.columns and weight_event is not None and weight_event > 1:
        ev = pd.to_numeric(pdf[event_col], errors="coerce").fillna(0).astype(np.int8).values
        w[ev == 1] = float(weight_event)
    return w

def train_best_model(train_X, train_y, test_X, test_y, n_grid, forced_best_n=None, sample_weight=None):
    candidates = [forced_best_n] if (forced_best_n is not None and forced_best_n > 0) else n_grid

    best_rmse_val = np.inf
    best_mae_val = None
    best_pred = None
    best_n = None

    for n_est in candidates:
        params = dict(base_xgb_params)
        params["n_estimators"] = int(n_est)

        model = XGBRegressor(**params)
        if sample_weight is not None:
            model.fit(train_X, train_y, sample_weight=sample_weight)
        else:
            model.fit(train_X, train_y)

        pred = model.predict(test_X).astype(np.float32)
        cur_rmse = rmse(test_y, pred)

        if cur_rmse < best_rmse_val:
            best_rmse_val = cur_rmse
            best_mae_val = float(mean_absolute_error(test_y, pred))
            best_pred = pred
            best_n = int(n_est)

    return best_pred, best_mae_val, best_rmse_val, best_n

def save_xgb_model_json_dbfs(model: XGBRegressor, dbfs_path: str):
    """
    Serverless-safe:
    - Export booster as raw JSON bytes
    - Atomic write as a single JSON file using dbutils.fs.put + mv
    """
    booster = model.get_booster()
    raw = booster.save_raw(raw_format="json")  # bytes
    json_str = raw.decode("utf-8")

    if len(json_str) < 1000:
        raise Exception(f"Model JSON suspiciously small before write: {dbfs_path}")

    atomic_put_text(dbfs_path, json_str)
    print(f"✅ Saved model JSON to: {dbfs_path} (bytes={len(json_str)})")

def save_json_dbfs(obj: dict, dbfs_path: str):
    json_str = json.dumps(obj, indent=2)
    if len(json_str) < 50:
        raise Exception(f"Metadata JSON suspiciously small before write: {dbfs_path}")

    atomic_put_text(dbfs_path, json_str)
    print(f"✅ Saved metadata JSON to: {dbfs_path} (bytes={len(json_str)})")

def load_booster_from_dbfs_json(dbfs_path: str, max_bytes: int = MAX_MODEL_BYTES) -> xgb.Booster:
    s = read_dbfs_text(dbfs_path, max_bytes=max_bytes)
    if len(s) < 1000:
        raise Exception(f"Model JSON looks too small/truncated: {dbfs_path} bytes_read={len(s)}")
    booster = xgb.Booster()
    booster.load_model(bytearray(s.encode("utf-8")))
    return booster

def validate_artifacts(dep_path: str, arr_path: str, meta_path: str, expected_rounds: int):
    # paths exist
    for p in [dep_path, arr_path, meta_path]:
        if not path_exists(p):
            raise Exception(f"Validation failed: missing artifact {p}")

    # meta loads
    meta = json.loads(read_dbfs_text(meta_path, max_bytes=MAX_META_BYTES))
    dep_feats = meta.get("dep_features", [])
    arr_feats = meta.get("arr_features", [])
    hb = int(meta.get("hash_buckets", -1))
    if not dep_feats or not arr_feats:
        raise Exception("Validation failed: metadata missing dep_features/arr_features")
    if len(set(dep_feats)) != len(dep_feats):
        raise Exception("Validation failed: dep_features has duplicates")
    if len(set(arr_feats)) != len(arr_feats):
        raise Exception("Validation failed: arr_features has duplicates")
    if hb <= 0:
        raise Exception("Validation failed: hash_buckets invalid")

    # boosters load + rounds
    dep_b = load_booster_from_dbfs_json(dep_path)
    arr_b = load_booster_from_dbfs_json(arr_path)

    dep_r = dep_b.num_boosted_rounds()
    arr_r = arr_b.num_boosted_rounds()

    print(f"✅ Validation: meta OK (dep_features={len(dep_feats)}, arr_features={len(arr_feats)}, hash_buckets={hb})")
    print(f"✅ Validation: boosters OK (dep_rounds={dep_r}, arr_rounds={arr_r})")

    if dep_r != expected_rounds:
        raise Exception(f"Validation failed: dep num_boosted_rounds={dep_r} expected={expected_rounds}")
    if arr_r != expected_rounds:
        raise Exception(f"Validation failed: arr num_boosted_rounds={arr_r} expected={expected_rounds}")

    print("🎉 ALL ARTIFACT VALIDATIONS PASSED")

# ------------------------------------------------------------
# 3) Load Features (GOLD_V2)
# ------------------------------------------------------------
if not path_exists(FEATURES_GOLDV2_DIR):
    raise Exception(f"FEATURES_GOLDV2_DIR not found: {FEATURES_GOLDV2_DIR}")

df_feat = spark.read.parquet(FEATURES_GOLDV2_DIR)

dep_target = "departures"
arr_target = "arrivals"

# ------------------------------------------------------------
# 4) Column definitions (GOLD_V2 schema)
# ------------------------------------------------------------
key_cols = ["station_id", "year", "month", "day", "hour", "date", "dow_num"]

dep_lag_cols  = ["lag1_dep", "lag2_dep", "lag24_dep", "lag168_dep"]
arr_lag_cols  = ["lag1_arr", "lag2_arr", "lag24_arr", "lag168_arr"]

dep_roll_cols = ["roll_mean_3h_dep", "roll_std_24h_dep"]
arr_roll_cols = ["roll_mean_3h_arr", "roll_std_24h_arr"]

weather_cols  = ["temperature_2m_celsius", "apparent_temperature_celsius"]

event_cols = [
    "event_day_flag",
    "events_day_count",
    "event_day_attendance_sum",
    EVENT_FLAG_COL,
    "events_nearby_count",
    "nearest_event_km",
    "event_weighted_intensity",
    "event_attendance_est_sum_nearby",
    "event_impact_score"
]

common_features = [
    BUCKET_COL, "month", "hour", "dow_num", "is_weekend",
    *weather_cols,
    *event_cols
]
dep_model_features = common_features + dep_lag_cols + dep_roll_cols
arr_model_features = common_features + arr_lag_cols + arr_roll_cols

# Freeze explicit order (deterministic)
dep_model_features = list(dep_model_features)
arr_model_features = list(arr_model_features)

# ------------------------------------------------------------
# 5) Validate columns exist (fail fast)
# ------------------------------------------------------------
cols_set = set(df_feat.columns)
must_have = (
    key_cols +
    [dep_target, arr_target] +
    dep_lag_cols + arr_lag_cols +
    dep_roll_cols + arr_roll_cols +
    weather_cols +
    event_cols
)
missing = [c for c in must_have if c not in cols_set]
if missing:
    raise Exception(f"GOLDV2 features dataset missing columns: {missing}")

print("✅ Column check passed (GOLD_V2 features)")

# ------------------------------------------------------------
# 6) Month list (ALL months in dataset)
# ------------------------------------------------------------
months_list = [(int(r["year"]), int(r["month"])) for r in (
    df_feat.select("year", "month").distinct().orderBy("year", "month").collect()
)]
print("Months to evaluate (raw):", len(months_list))
print("First/Last month:", months_list[0] if months_list else None, months_list[-1] if months_list else None)

# ------------------------------------------------------------
# 7) Optional: Load best_n per month (dep/arr)
# ------------------------------------------------------------
best_n_dep = {}
best_n_arr = {}

if USE_BEST_N_FROM_PREV_RUNS:
    if not path_exists(EVAL_DEP_DIR):
        raise Exception(f"EVAL_DEP_DIR not found: {EVAL_DEP_DIR}")
    if not path_exists(EVAL_ARR_DIR):
        raise Exception(f"EVAL_ARR_DIR not found: {EVAL_ARR_DIR}")

    dep_prev = spark.read.parquet(EVAL_DEP_DIR).toPandas()
    arr_prev = spark.read.parquet(EVAL_ARR_DIR).toPandas()

    for _, r in dep_prev.iterrows():
        best_n_dep[(int(r["year"]), int(r["month"]))] = int(r.get("best_n_estimators", -1))
    for _, r in arr_prev.iterrows():
        best_n_arr[(int(r["year"]), int(r["month"]))] = int(r.get("best_n_estimators", -1))

    print("Loaded per-month best_n from previous runs")

# ------------------------------------------------------------
# 8) Pandas month cache (1 toPandas por mes)
# ------------------------------------------------------------
month_cache = {}

month_select_cols = (
    key_cols +
    [dep_target, arr_target] +
    dep_lag_cols + arr_lag_cols +
    dep_roll_cols + arr_roll_cols +
    weather_cols + event_cols
)

numeric_cols = weather_cols + event_cols + dep_lag_cols + arr_lag_cols + dep_roll_cols + arr_roll_cols

def load_month(y: int, m: int) -> pd.DataFrame:
    key = (y, m)
    if key in month_cache:
        return month_cache[key]

    sdf = (
        df_feat
        .filter((F.col("year") == y) & (F.col("month") == m))
        .select(month_select_cols)
    )

    pdf = sdf.toPandas()
    if len(pdf) > 0:
        pdf["date"] = pd.to_datetime(pdf["date"])
        pdf[BUCKET_COL] = pdf["station_id"].map(lambda s: station_to_bucket(s, HASH_BUCKETS)).astype(np.int16)
        pdf["is_weekend"] = pdf["dow_num"].isin([1, 7]).astype(np.int8)

        for c in numeric_cols:
            if c in pdf.columns:
                pdf[c] = pd.to_numeric(pdf[c], errors="coerce").fillna(0.0)

    month_cache[key] = pdf
    return pdf

# ------------------------------------------------------------
# 9) Monthly loop: train dep + arr, derive net_flow, evaluate
# ------------------------------------------------------------
results = []
skipped_no_train = 0

for (y, m) in months_list:
    test_start = pd.Timestamp(year=y, month=m, day=1)
    test_end = test_start + pd.offsets.MonthEnd(0)

    train_end = test_start - pd.Timedelta(days=1)
    train_start = train_end - pd.Timedelta(days=TRAIN_LOOKBACK_DAYS)

    test_pd = load_month(y, m)
    if test_pd.empty:
        continue

    train_parts = []
    for (yy, mm) in prev_months_in_lookback(test_start, TRAIN_LOOKBACK_DAYS):
        part = load_month(yy, mm)
        if not part.empty:
            train_parts.append(part)

    if not train_parts:
        skipped_no_train += 1
        continue

    train_all = pd.concat(train_parts, ignore_index=True)

    train_pd = train_all[(train_all["date"] >= train_start) & (train_all["date"] <= train_end)]
    test_pd_f = test_pd[(test_pd["date"] >= test_start) & (test_pd["date"] <= test_end)]

    if train_pd.empty or test_pd_f.empty:
        skipped_no_train += 1
        continue

    join_keys = ["station_id", "date", "hour"]
    train_pd = train_pd.sort_values(join_keys).reset_index(drop=True)
    test_pd_f = test_pd_f.sort_values(join_keys).reset_index(drop=True)

    dep_train_X, dep_train_y = build_xy(train_pd, dep_model_features, dep_target)
    dep_test_X, dep_test_y   = build_xy(test_pd_f, dep_model_features, dep_target)

    arr_train_X, arr_train_y = build_xy(train_pd, arr_model_features, arr_target)
    arr_test_X, arr_test_y   = build_xy(test_pd_f, arr_model_features, arr_target)

    train_w = build_sample_weights(train_pd, WEIGHT_EVENT, event_col=EVENT_FLAG_COL)

    forced_dep_n = best_n_dep.get((y, m), None) if USE_BEST_N_FROM_PREV_RUNS else None
    forced_arr_n = best_n_arr.get((y, m), None) if USE_BEST_N_FROM_PREV_RUNS else None

    dep_pred, _, _, dep_best_n = train_best_model(
        dep_train_X, dep_train_y, dep_test_X, dep_test_y,
        N_ESTIMATORS_GRID, forced_best_n=forced_dep_n, sample_weight=train_w
    )
    arr_pred, _, _, arr_best_n = train_best_model(
        arr_train_X, arr_train_y, arr_test_X, arr_test_y,
        N_ESTIMATORS_GRID, forced_best_n=forced_arr_n, sample_weight=train_w
    )

    net_real = (arr_test_y - dep_test_y).astype(np.float32)
    net_pred = (arr_pred - dep_pred).astype(np.float32)

    baseline_net = (
        test_pd_f["lag1_arr"].astype(np.float32).values
        - test_pd_f["lag1_dep"].astype(np.float32).values
    )

    baseline_mae = float(mean_absolute_error(net_real, baseline_net))
    model_mae    = float(mean_absolute_error(net_real, net_pred))
    baseline_rmse_val = rmse(net_real, baseline_net)
    model_rmse_val    = rmse(net_real, net_pred)

    improvement_pct = (baseline_mae - model_mae) / baseline_mae * 100 if baseline_mae else np.nan

    # Segmented metrics
    event_flag = pd.to_numeric(test_pd_f[EVENT_FLAG_COL], errors="coerce").fillna(0).astype(np.int8).values
    idx_event = (event_flag == 1)
    idx_noev  = (event_flag == 0)

    seg = {}
    seg["rows_event"] = int(idx_event.sum())
    seg["rows_no_event"] = int(idx_noev.sum())

    if idx_event.sum() > 0:
        seg["baseline_mae_net_event"]   = float(mean_absolute_error(net_real[idx_event], baseline_net[idx_event]))
        seg["model_mae_net_event"]      = float(mean_absolute_error(net_real[idx_event], net_pred[idx_event]))
        seg["baseline_rmse_net_event"]  = rmse(net_real[idx_event], baseline_net[idx_event])
        seg["model_rmse_net_event"]     = rmse(net_real[idx_event], net_pred[idx_event])
    else:
        seg["baseline_mae_net_event"]   = np.nan
        seg["model_mae_net_event"]      = np.nan
        seg["baseline_rmse_net_event"]  = np.nan
        seg["model_rmse_net_event"]     = np.nan

    if idx_noev.sum() > 0:
        seg["baseline_mae_net_no_event"]  = float(mean_absolute_error(net_real[idx_noev], baseline_net[idx_noev]))
        seg["model_mae_net_no_event"]     = float(mean_absolute_error(net_real[idx_noev], net_pred[idx_noev]))
        seg["baseline_rmse_net_no_event"] = rmse(net_real[idx_noev], baseline_net[idx_noev])
        seg["model_rmse_net_no_event"]    = rmse(net_real[idx_noev], net_pred[idx_noev])
    else:
        seg["baseline_mae_net_no_event"]  = np.nan
        seg["model_mae_net_no_event"]     = np.nan
        seg["baseline_rmse_net_no_event"] = np.nan
        seg["model_rmse_net_no_event"]    = np.nan

    results.append({
        "year": y,
        "month": m,
        "rows_test": int(len(net_real)),
        "baseline_mae_net": baseline_mae,
        "model_mae_net": model_mae,
        "baseline_rmse_net": baseline_rmse_val,
        "model_rmse_net": model_rmse_val,
        "improvement_pct_net": float(improvement_pct),
        "dep_best_n_estimators": int(dep_best_n),
        "arr_best_n_estimators": int(arr_best_n),
        "hash_buckets": int(HASH_BUCKETS),
        "train_lookback_days": int(TRAIN_LOOKBACK_DAYS),
        "weight_event": float(WEIGHT_EVENT),
        **seg
    })

print(f"Done. results months={len(results)} skipped_no_train={skipped_no_train}")

# ------------------------------------------------------------
# 10) Save monthly metrics parquet
# ------------------------------------------------------------
results_pd = pd.DataFrame(results)
display(results_pd)

results_spark = spark.createDataFrame(results_pd)
(results_spark.write.mode("overwrite").parquet(EVAL_DIR_NETFLOW))
print("✅ Saved netflow eval parquet to:", EVAL_DIR_NETFLOW)

# ============================================================
# 11) Train FINAL serving models (dep + arr) on ALL data
# ============================================================
print("\n==================== TRAIN FINAL SERVING MODELS ====================")

all_parts = []
for (yy, mm) in months_list:
    part = load_month(yy, mm)
    if not part.empty:
        all_parts.append(part)

if not all_parts:
    raise Exception("No data loaded to train final models.")

full_pd = pd.concat(all_parts, ignore_index=True)
full_pd = full_pd.sort_values(["station_id", "date", "hour"]).reset_index(drop=True)

dep_X, dep_y = build_xy(full_pd, dep_model_features, dep_target)
arr_X, arr_y = build_xy(full_pd, arr_model_features, arr_target)

full_w = build_sample_weights(full_pd, WEIGHT_EVENT, event_col=EVENT_FLAG_COL)

FINAL_N_ESTIMATORS = 900

dep_params = dict(base_xgb_params)
dep_params["n_estimators"] = FINAL_N_ESTIMATORS

arr_params = dict(base_xgb_params)
arr_params["n_estimators"] = FINAL_N_ESTIMATORS

dep_model = XGBRegressor(**dep_params)
arr_model = XGBRegressor(**arr_params)

dep_model.fit(dep_X, dep_y, sample_weight=full_w)
arr_model.fit(arr_X, arr_y, sample_weight=full_w)

# ---- Save models as JSON (atomic, serverless-safe)
ensure_dir_dbfs(SERVING_MODEL_DIR_DBFS)
save_xgb_model_json_dbfs(dep_model, DEP_MODEL_JSON_DBFS)
save_xgb_model_json_dbfs(arr_model, ARR_MODEL_JSON_DBFS)

# ---- Save metadata JSON (feature contract)
meta = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "weight_event": WEIGHT_EVENT,
    "hash_buckets": HASH_BUCKETS,
    "final_n_estimators": FINAL_N_ESTIMATORS,
    "event_flag_col": EVENT_FLAG_COL,
    "dep_features": dep_model_features,
    "arr_features": arr_model_features,
    "numeric_cols": numeric_cols,
}

ensure_dir_dbfs(FEATURE_META_DIR_DBFS)
save_json_dbfs(meta, FEATURE_META_JSON_DBFS)

print("\n✅ FINAL SERVING ARTIFACTS (DBFS)")
print("DEP JSON:", DEP_MODEL_JSON_DBFS)
print("ARR JSON:", ARR_MODEL_JSON_DBFS)
print("META JSON:", FEATURE_META_JSON_DBFS)

# ---- Post-write validation (catches truncation / partial files)
print("\n==================== POST-WRITE VALIDATION ====================")
validate_artifacts(
    dep_path=DEP_MODEL_JSON_DBFS,
    arr_path=ARR_MODEL_JSON_DBFS,
    meta_path=FEATURE_META_JSON_DBFS,
    expected_rounds=FINAL_N_ESTIMATORS
)

print("\nℹ️ JOB B should load using these filesystem paths (for debugging only):")
print("DEP:", "/dbfs" + DEP_MODEL_JSON_DBFS.replace("dbfs:", ""))
print("ARR:", "/dbfs" + ARR_MODEL_JSON_DBFS.replace("dbfs:", ""))
print("META:", "/dbfs" + FEATURE_META_JSON_DBFS.replace("dbfs:", ""))

**Artifact Validation**

This section verifies that trained models and feature metadata have been correctly stored and can be loaded for prediction.

Basic validation tests are performed to ensure readiness for deployment.

In [0]:
import json
import xgboost as xgb

WEIGHT_EVENT = 10
DEP_MODEL_DBFS = f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/models/dep_weight{WEIGHT_EVENT}.json"
ARR_MODEL_DBFS = f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/models/arr_weight{WEIGHT_EVENT}.json"
META_DBFS      = f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/metadata/features_weight{WEIGHT_EVENT}/features.json"

def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def read_dbfs_text(dbfs_path: str, max_bytes: int) -> str:
    return dbutils.fs.head(dbfs_path, max_bytes)

def load_booster_from_dbfs_json(dbfs_path: str, max_bytes: int = 25_000_000) -> xgb.Booster:
    s = read_dbfs_text(dbfs_path, max_bytes=max_bytes)
    booster = xgb.Booster()
    booster.load_model(bytearray(s.encode("utf-8")))
    return booster

print("=== VALIDATE MODEL ARTIFACTS ===")

# 1) Paths exist
for p in [DEP_MODEL_DBFS, ARR_MODEL_DBFS, META_DBFS]:
    if not path_exists(p):
        raise Exception(f"Missing artifact: {p}")
print("✅ DBFS paths exist")

# 2) Metadata contract
meta = json.loads(read_dbfs_text(META_DBFS, max_bytes=300_000))
dep_feats = meta["dep_features"]
arr_feats = meta["arr_features"]
hash_buckets = int(meta.get("hash_buckets", 512))

if not isinstance(dep_feats, list) or not isinstance(arr_feats, list):
    raise Exception("dep_features/arr_features must be lists")
if len(dep_feats) == 0 or len(arr_feats) == 0:
    raise Exception("Feature lists are empty")

print(f"✅ Metadata loaded. dep_features={len(dep_feats)}, arr_features={len(arr_feats)}, hash_buckets={hash_buckets}")

# 3) Load boosters + sanity checks
dep_booster = load_booster_from_dbfs_json(DEP_MODEL_DBFS)
arr_booster = load_booster_from_dbfs_json(ARR_MODEL_DBFS)

dep_rounds = dep_booster.num_boosted_rounds()
arr_rounds = arr_booster.num_boosted_rounds()

print("✅ Boosters loaded in memory")
print("dep num_boosted_rounds:", dep_rounds)
print("arr num_boosted_rounds:", arr_rounds)

if dep_rounds <= 0 or arr_rounds <= 0:
    raise Exception("Invalid boosters: num_boosted_rounds is 0")

# 4) Quick smoke prediction (shape only)
import numpy as np
X_dep = np.zeros((2, len(dep_feats)), dtype=np.float32)
X_arr = np.zeros((2, len(arr_feats)), dtype=np.float32)

dep_pred = dep_booster.predict(xgb.DMatrix(X_dep))
arr_pred = arr_booster.predict(xgb.DMatrix(X_arr))

if np.any(np.isnan(dep_pred)) or np.any(np.isnan(arr_pred)):
    raise Exception("NaNs in smoke predictions")

print("✅ Smoke prediction OK (no NaNs)")
print("🎉 MODEL ARTIFACTS LOOK PERFECT (contract + boosted rounds + smoke predict)")

**Results and Interpretation**

The models capture temporal demand patterns for both departures and arrivals using XGBoost, enabling the derivation of net flow as a measure of station imbalance.

This approach provides a comprehensive representation of system dynamics and serves as the foundation for downstream prediction and decision-making processes.
